# A small CNN on CPU

Train a convolutional classifier on built-in 8×8 digit images. This lesson makes convolution shapes, pooling, validation-based epoch selection, and held-out evaluation visible without downloading MNIST or pretrained weights.

**Prerequisites:** PyTorch tensors and training loops. **Environment:** base requirements plus CPU PyTorch. **Execution:** restart and run all cells. No external dataset, torchvision, or GPU is required.


In [ ]:
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
DEVICE = torch.device('cpu')
print('PyTorch:', torch.__version__, 'Device:', DEVICE)


## Prepare images in channel-first order

PyTorch convolutions expect `(batch, channels, height, width)`. Dividing by the documented pixel maximum of 16 is a fixed transformation; it learns no statistics from the test data.


In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development, y_development, test_size=0.2, stratify=y_development, random_state=SEED
)
print('Train / validation / test:', len(y_train), len(y_validation), len(y_test))
X_train = (X_train / 16).reshape(-1, 1, 8, 8)
X_validation = (X_validation / 16).reshape(-1, 1, 8, 8)
X_test = (X_test / 16).reshape(-1, 1, 8, 8)
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for index, ax in enumerate(axes):
    ax.imshow(X_train[index, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(str(y_train[index]))
    ax.axis('off')
plt.show()


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
                                      nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
                                      nn.MaxPool2d(2))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(16 * 4 * 4, 32),
                                        nn.ReLU(), nn.Linear(32, 10))
    def forward(self, images):
        return self.classifier(self.features(images))

torch.manual_seed(SEED)
model = SmallCNN().to(DEVICE)
probe = torch.tensor(X_train[:4])
assert model.features(probe).shape == (4, 16, 4, 4)
assert model(probe).shape == (4, 10)
print('Trainable parameters:', sum(parameter.numel() for parameter in model.parameters()))


## Train on training data and select on validation data

Each batch clears accumulated gradients, computes the loss, backpropagates, and updates parameters. `train()` enables training behavior; `eval()` and `no_grad()` are used for evaluation. The helper below retains a **copy** of the weights with the lowest validation loss. The test set is not used to choose an epoch or model.

Seeds make this small CPU experiment repeatable in the same environment. Bitwise equality across different library versions or hardware is not promised.


In [ ]:
def loader(features, targets, shuffle=False, batch_size=64):
    dataset = TensorDataset(torch.as_tensor(features, dtype=torch.float32),
                            torch.as_tensor(targets, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))

def evaluate_classifier(model, batches):
    model.eval()  # disables dropout and uses inference behavior
    total_loss, total, correct = 0.0, 0, 0
    all_predictions = []
    with torch.no_grad():  # validation does not build a gradient graph
        for features, targets in batches:
            logits = model(features.to(DEVICE))
            targets = targets.to(DEVICE)
            total_loss += nn.functional.cross_entropy(logits, targets, reduction='sum').item()
            prediction = logits.argmax(dim=1)
            correct += (prediction == targets).sum().item()
            total += targets.numel()
            all_predictions.append(prediction.cpu().numpy())
    return total_loss / total, correct / total, np.concatenate(all_predictions)

def fit_classifier(model, train_batches, validation_batches, epochs=10, learning_rate=0.005):
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history, best_state, best_validation_loss = [], None, float('inf')
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, total = 0.0, 0
        for features, targets in train_batches:
            features, targets = features.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(features), targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * targets.numel()
            total += targets.numel()
        validation_loss, validation_accuracy, _ = evaluate_classifier(model, validation_batches)
        history.append({'epoch': epoch, 'train_loss': total_loss / total,
                        'validation_loss': validation_loss,
                        'validation_accuracy': validation_accuracy})
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    model.eval()
    return pd.DataFrame(history)


In [ ]:
history = fit_classifier(model, loader(X_train, y_train, shuffle=True),
                         loader(X_validation, y_validation), epochs=10, learning_rate=0.005)
history.plot(x='epoch', y=['train_loss', 'validation_loss'], figsize=(6, 3))
plt.ylabel('Cross-entropy')
plt.show()
display(history.tail())


## Test the selected weights

The held-out test set has not chosen the architecture or checkpoint. A simple dataset can produce encouraging scores with a small model; deployment to different cameras, handwriting styles, or resolutions requires separate evidence.


In [ ]:
test_loss, test_accuracy, predictions = evaluate_classifier(model, loader(X_test, y_test))
print('Held-out loss:', round(test_loss, 4), 'Accuracy:', round(test_accuracy, 4))
ConfusionMatrixDisplay.from_predictions(y_test, predictions)
plt.show()
wrong = np.flatnonzero(predictions != y_test)[:6]
if len(wrong):
    fig, axes = plt.subplots(1, len(wrong), figsize=(2 * len(wrong), 2), squeeze=False)
    for ax, index in zip(axes[0], wrong):
        ax.imshow(X_test[index, 0], cmap='gray', vmin=0, vmax=1)
        ax.set_title(f'True {y_test[index]} / predicted {predictions[index]}')
        ax.axis('off')
    plt.show()


## Practice

Remove one convolution or replace max pooling with average pooling. Record parameter counts and validation results before checking a fresh held-out set.

**References:** [PyTorch Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html), [scikit-learn digits](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html).
